In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [3]:
df=pd.read_csv('E:\python project\project5\imdb_movies_2024(1).csv')
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\HP\AppData\Local\Temp\ipykernel_13996\2559869306.py:1: SyntaxWarning: invalid escape sequence '\p'
  df=pd.read_csv('E:\python project\project5\imdb_movies_2024(1).csv')


,Title,Story_line
0,1. The Substance,A fading celebrity takes a black-market drug: ...
1,2. The Life of Chuck,"A life-affirming, genre-bending story about th..."
2,3. Bone Lake,A couple's vacation at a secluded estate is up...
3,4. Anora,A young stripper from Brooklyn meets and impul...
4,5. Eden,Based on a factual account of a group of outsi...


In [4]:
pip install nltk contractions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import re
import nltk
import contractions

In [6]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [7]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [8]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [9]:
def text_preprocessing(text):
    text = str(text).lower()
    text = contractions.fix(text)
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation & numbers

    tokens = word_tokenize(text)

    processed_text = ' '.join(
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    )

    return processed_text

In [10]:
df['clean_storyline'] = df['Story_line'].apply(text_preprocessing)

In [11]:
df[['Title', 'Story_line', 'clean_storyline']].head()

,Title,Story_line,clean_storyline
0,1. The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity take blackmarket drug cellrep...
1,2. The Life of Chuck,"A life-affirming, genre-bending story about th...",lifeaffirming genrebending story three chapter...
2,3. Bone Lake,A couple's vacation at a secluded estate is up...,couple vacation secluded estate upended forced...
3,4. Anora,A young stripper from Brooklyn meets and impul...,young stripper brooklyn meet impulsively marri...
4,5. Eden,Based on a factual account of a group of outsi...,based factual account group outsider settle re...


In [12]:
df['Title'] = df['Title'].str.replace(r'^\d+\.\s*', '', regex=True)

In [13]:
df[['Title', 'Story_line', 'clean_storyline']].head()

,Title,Story_line,clean_storyline
0,The Substance,A fading celebrity takes a black-market drug: ...,fading celebrity take blackmarket drug cellrep...
1,The Life of Chuck,"A life-affirming, genre-bending story about th...",lifeaffirming genrebending story three chapter...
2,Bone Lake,A couple's vacation at a secluded estate is up...,couple vacation secluded estate upended forced...
3,Anora,A young stripper from Brooklyn meets and impul...,young stripper brooklyn meet impulsively marri...
4,Eden,Based on a factual account of a group of outsi...,based factual account group outsider settle re...


In [14]:
df.shape

(1589, 3)

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df.isnull().sum()

Title              0
Story_line         0
clean_storyline    0
dtype: int64

In [17]:
df.dtypes

Title              object
Story_line         object
clean_storyline    object
dtype: object

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df['clean_storyline'])

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
cosine_sim=cosine_similarity(tfidf_matrix,tfidf_matrix)

In [22]:
def recommand_movies(input_storyline,df,tfidf,tfidf_matrix,top_n=5):
  clean_storyline=text_preprocessing(input_storyline)
  input_vector=tfidf.transform([clean_storyline])
  similarity_scores=cosine_similarity(input_vector,tfidf_matrix).flatten()
  top_indices=similarity_scores.argsort()[-top_n:][::-1]
  return df.iloc[top_indices][['Title','Story_line']]

In [23]:
text="A young magician attends a magic school and faces dark forces"
recommand_movies(text,df,tfidf,tfidf_matrix)

,Title,Story_line
1360,Heresy,"In a medieval Dutch village, a young woman is ..."
192,Witchboard,"A cursed Witchboard awakens dark forces, dragg..."
184,Apartment 7A,A struggling young dancer finds herself drawn ...
1449,Strange Frequencies: Taiwan Killer Hospital,Reality TV stars face mounting supernatural ho...
1243,The Curse of the Necklace,Family navigates turbulent 1960s dynamics. Mom...
